# 1. Exercise

### Download and Transform the Data
The code here has been copied from the Chapter 1 Notebook.

In [1]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans # https://scikit-learn.org/1.5/modules/generated/sklearn.cluster.KMeans.html
from sklearn.metrics.pairwise import rbf_kernel

""" 
    We inherit from these two classes because they give us a few things that we then don't have to implement.
    BaseEstimator - gives us the get_params and set_params that allows us to easily tune our hyper parameters.
    TransformerMixin - gives us the fit_transform method for free (it simply calls fit and then transform)
"""
class ClusterSimilarity(BaseEstimator, TransformerMixin):

    # The args here are because we are using KMeans. n_clusters=10 means we find 10 clusters.
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    # we need to implement our fit and transform and fit should return self.
    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    # This method is helpful when you want your output to be a dataframe
    def get_feature_names_out(self, names=None):
        return [f"Clusters {i} similarity" for i in range(self.n_clusters)]

In [3]:
from sklearn.model_selection import train_test_split

housing = load_housing_data()
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X,y , random_state=104, test_size=0.25, shuffle=True)

In [4]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"), # fill empty values with the median
        FunctionTransformer(column_ratio, feature_names_out=ratio_name), # calculate the ratios
        StandardScaler() # scale the values using standardization
    )

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"), # fill the empty values with median
    FunctionTransformer(np.log, feature_names_out="one-to-one"), # apply a log function to reduce the heavy tail
    StandardScaler() # scale them using standardization
)

# write a pipeline to handle categorical values
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"), # fill the empty values with the most frequent
    OneHotEncoder(handle_unknown="ignore"), # one hot encode the categories so that the model does not have to deal with word representations
)
cluster_simil = ClusterSimilarity(random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

In [5]:
from sklearn.compose import ColumnTransformer, make_column_selector

preprocessing = ColumnTransformer([
   ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]), # calculate the ration of the two
   ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
   ("people_per_house", ratio_pipeline(), ["population", "households"]),
   ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
   ("geo", cluster_simil, ["latitude", "longitude"]),
   ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

## Question 1
Exercise: Try a Support Vector Machine regressor (`sklearn.svm.SVR`) with various hyperparameters, such as `kernel="linear"` (with various values for the `C` hyperparameter) or `kernel="rbf"` (with various values for the `C` and `gamma` hyperparameters). Note that SVMs don't scale well to large datasets, so you should probably train your model on just the first 5,000 instances of the training set and use only 3-fold cross-validation, or else it will take hours. Don't worry about what the hyperparameters mean for now (see the SVM notebook if you're interested). How does the best SVR predictor perform?

In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR

In [7]:
svm_pipeline = make_pipeline(preprocessing, SVR())
svm_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_name at 0x159...
                                                   'total_rooms', 'population',
                                                   'households',
                                                   'median_income']),
                                                 ('geo',
                                                  ClusterSimilarity(random_state=42),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x15a3873e0>)])),
                ('svr', SVR())])

In [8]:
grid_search_parameters = [
    {"svr__kernel": ["linear"], "svr__C": (1.0, 100, 300, 1000), "svr__epsilon": [0.1, 0.3, 0.5, 0.8]},
    {"svr__kernel": ["rbf"], "svr__C": (100, 300, 1000), "svr__epsilon": [0.1, 0.3, 0.5, 0.8], "svr__gamma": [0.1, 0.3, 0.5, 0.8]}   
]
gs = GridSearchCV(svm_pipeline, grid_search_parameters, scoring="neg_root_mean_squared_error", cv=3)

In [9]:
%%time
gs = gs.fit(X_train[:5000], y_train[:5000])

CPU times: user 3min 43s, sys: 2min 35s, total: 6min 18s
Wall time: 1min 20s


In [10]:
import pandas as pd

results_df = pd.DataFrame(data=gs.cv_results_)
results_df[["rank_test_score", "param_svr__kernel", "param_svr__C", "param_svr__epsilon", "mean_test_score"]].sort_values(["rank_test_score"])[:10]

,rank_test_score,param_svr__kernel,param_svr__C,param_svr__epsilon,mean_test_score
60,1,rbf,1000.0,0.8,-84300.856680
56,2,rbf,1000.0,0.5,-84300.864104
52,3,rbf,1000.0,0.3,-84300.869083
48,4,rbf,1000.0,0.1,-84300.874068
15,5,linear,1000.0,0.8,-88373.083376
14,6,linear,1000.0,0.5,-88373.343184
13,7,linear,1000.0,0.3,-88373.516134
12,8,linear,1000.0,0.1,-88373.689351
61,9,rbf,1000.0,0.8,-94108.511435
57,10,rbf,1000.0,0.5,-94108.513240


Based on the above results, its seems like the linear kernel functions much better than the rbf one. However the performance for it is still pretty bad.
Lets try running it on the test set and see what we get.

In [11]:
gs.best_params_

{'svr__C': 1000, 'svr__epsilon': 0.8, 'svr__gamma': 0.1, 'svr__kernel': 'rbf'}

## Question 2
Try replacing the `GridSearchCV` with a `RandomizedSearchCV`

In [12]:
from sklearn.model_selection import RandomizedSearchCV

In [13]:
from scipy.stats import expon,loguniform

randomized_search_parameters = [
    {"svr__kernel": ["linear"], "svr__C": loguniform(20, 200_000)},
    {"svr__kernel": ["rbf"], "svr__C": loguniform(20, 200_000), "svr__gamma": expon(scale=1.0)}
]
rs = RandomizedSearchCV(svm_pipeline, randomized_search_parameters, scoring="neg_root_mean_squared_error", cv=3)

In [14]:
%%time
rs = rs.fit(X_train[:5000], y_train[:5000])

CPU times: user 45.6 s, sys: 25.1 s, total: 1min 10s
Wall time: 20.2 s


In [15]:
import pandas as pd

results_df = pd.DataFrame(data=rs.cv_results_)
results_df[["rank_test_score", "param_svr__kernel", "param_svr__C", "mean_test_score"]].sort_values(["rank_test_score"])[:10]

,rank_test_score,param_svr__kernel,param_svr__C,mean_test_score
9,1,rbf,137231.563928,-57337.971746
5,2,linear,2579.883358,-72567.380268
8,3,linear,5118.156104,-78341.126295
6,4,linear,5679.881430,-78503.350961
3,5,linear,12125.857756,-79056.618822
7,6,rbf,182141.470931,-82785.821633
0,7,rbf,2042.440571,-110420.303478
4,8,rbf,132.915600,-110687.958607
1,9,linear,145105.683369,-110849.721475
2,10,rbf,268.666648,-115924.009679


In [16]:
rs.best_params_F

{'svr__C': 137231.56392840215,
 'svr__gamma': 0.3843763792260751,
 'svr__kernel': 'rbf'}

In [17]:
best_arguments = rs.best_params_
svr_best = SVR(kernel=best_arguments["svr__kernel"], C=best_arguments["svr__C"])
svm_tweaked_pipeline = make_pipeline(preprocessing, svr_best)
svm_tweaked_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_name at 0x159...
                                                   'households',
                                                   'median_income']),
                                                 ('geo',
                                                  ClusterSimilarity(random_state=42),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x15a3873e0>)])),
                ('svr', SVR(C=137231.56392840215))])

In [18]:
from sklearn.model_selection import cross_val_score

selector_rmses = -cross_val_score(svm_tweaked_pipeline,
                                  X_train,
                                  y_train,
                                  scoring="neg_root_mean_squared_error",
                                  cv=10)
pd.Series(selector_rmses).describe()

count       10.000000
mean     54476.947059
std       3285.872892
min      50850.838842
25%      52135.767115
50%      53352.368940
75%      56146.787542
max      61741.746315
dtype: float64

## Question 3
Try adding a `SelectFromModel` transformer in the preparation pipeline to select only the most important attributes

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

In [20]:
sm_pipeline = make_pipeline(preprocessing, SelectFromModel(RandomForestRegressor(random_state=42), threshold=0.005), svr_best)

In [21]:
selector_rmses = -cross_val_score(sm_pipeline,
                                  X_train,
                                  y_train,
                                  scoring="neg_root_mean_squared_error",
                                  cv=10)
pd.Series(selector_rmses).describe()

count       10.000000
mean     54960.956693
std       3373.587487
min      50494.537621
25%      52861.204355
50%      54050.811027
75%      56581.320301
max      62093.768695
dtype: float64

## Question 4
Try creating a custom transformer that trains a k-nearest neighbors regressor (`sklearn.neighbors.KNeighborsRegressor`) in its `fit()` method, and outputs
the model's prediction in its `transform()` method. Then add this feature to the preprocessing pipeline, using latitude and longitude as the inputs to this transformer. This will add a feature in the model that corresponds to the housing median price of the nearest districts.

In [22]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import KNeighborsRegressor
from sklearn.utils.validation import check_array, check_is_fitted

In [23]:
X.shape

(20640, 9)

In [24]:
class CustomNeighborEstimator(BaseEstimator, TransformerMixin):

    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y=None):
        X = check_array(X)
        self._kn = KNeighborsRegressor(n_neighbors=self.n_neighbors)
        self._kn.fit(X, y)
        self.n_features_in_ = self._kn.n_features_in_
        if hasattr(self._kn, "feature_names_in_"):
            self.feature_names_in_ = self._kn.feature_names_in_
        return self

    def transform(self, X):
        predictions = self._kn.predict(X)
        if predictions.ndim == 1:
            predictions = predictions.reshape(-1, 1)
        return predictions

    def get_feature_names_out(self, names=None):
        check_is_fitted(self)
        n_outputs = getattr(self._kn, "n_outputs_", 1)
        return [f"kn_prediction_{i}" for i in range(n_outputs)]

custom_estimator = CustomNeighborEstimator()

In [25]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.base import MetaEstimatorMixin, clone

class FeatureFromRegressor(MetaEstimatorMixin, BaseEstimator, TransformerMixin):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y=None):
        estimator_ = clone(self.estimator)
        estimator_.fit(X, y)
        self.estimator_ = estimator_
        self.n_features_in_ = self.estimator_.n_features_in_
        if hasattr(self.estimator, "feature_names_in_"):
            self.feature_names_in_ = self.estimator.feature_names_in_
        return self  # always return self!
    
    def transform(self, X):
        check_is_fitted(self)
        predictions = self.estimator_.predict(X)
        if predictions.ndim == 1:
            predictions = predictions.reshape(-1, 1)
        return predictions

    def get_feature_names_out(self, names=None):
        check_is_fitted(self)
        n_outputs = getattr(self.estimator_, "n_outputs_", 1)
        estimator_class_name = self.estimator_.__class__.__name__
        estimator_short_name = estimator_class_name.lower().replace("_", "")
        return [f"{estimator_short_name}_prediction_{i}"
                for i in range(n_outputs)]

In [26]:
knn_reg = KNeighborsRegressor(n_neighbors=3, weights="distance")
knn_transformer = FeatureFromRegressor(knn_reg)

In [32]:
from sklearn.pipeline import Pipeline

custom_estimator_preprocessing = ColumnTransformer([
   ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]), # calculate the ration of the two
   ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
   ("people_per_house", ratio_pipeline(), ["population", "households"]),
   ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
   ("geo", knn_transformer, ["latitude", "longitude"]),
   ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)
custom_estimator_pipeline = Pipeline([("preprocessing", custom_estimator_preprocessing), ("svr", svr_best)])
custom_estimator_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_name at 0x159fdc7...
                                                  FeatureFromRegressor(estimator=KNeighborsRegressor(n_neighbors=3,
                                                                                                     weights='distance')),
                                                  ['latitude', 'longitude']),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x15bbc4170>)])),
                ('svr', SVR(C=137231.56392840215))])

In [ ]:
custom_estimator_pipeline = custom_estimator_pipeline.fit(X[0:5000], y[0:5000])

In [29]:
%%time
selector_rmses = -cross_val_score(custom_estimator_pipeline,
                                  X_train,
                                  y_train,
                                  scoring="neg_root_mean_squared_error",
                                  cv=10)
pd.Series(selector_rmses).describe()

CPU times: user 37 s, sys: 434 ms, total: 37.4 s
Wall time: 37.7 s


count        10.000000
mean     105025.793854
std        2669.703326
min      102614.613197
25%      103154.548685
50%      104383.259865
75%      105188.248756
max      111000.243332
dtype: float64

## Question 5
Automatically explore some preparation options using GridSearchCV

In [33]:
grid_search_parameters = {
    "svr__kernel": ["linear"],
    "svr__C": (1000,),
    "preprocessing__geo__estimator__n_neighbors": [3]
}
gs = GridSearchCV(xx, grid_search_parameters, scoring="neg_root_mean_squared_error", cv=2)

In [ ]:
%%time
gs = gs.fit(X_train[:5000], y_train[:5000])

In [32]:
import pandas as pd

results_df = pd.DataFrame(data=rs.cv_results_)
results_df[["rank_test_score", "param_svr__kernel", "param_svr__C", "mean_test_score"]].sort_values(["rank_test_score"])[:10]

,rank_test_score,param_svr__kernel,param_svr__C,mean_test_score
2,1,rbf,5590.222462,-67469.769087
0,2,linear,2333.157849,-71657.932420
6,3,linear,16088.822307,-79264.584367
5,4,rbf,1075.435150,-86149.871975
3,5,linear,30.629953,-108502.041378
9,6,linear,323.474149,-113582.048327
1,7,linear,264.463409,-114876.685633
7,8,rbf,150.925627,-117098.004159
4,9,linear,197014.038345,-117131.256997
8,10,rbf,22.365938,-117359.660278


In [33]:
print("text")

text


## Question 6
Try to implement the StandardScalerClone class again from scratch, then add support for the `inverse_transform()` method: executing `scaler.inverse_transform(scaler.fit_transform(X))` should return an array 
very close to X. Then add support for feature names: set `feature_names_in_` in the `fit()` method if the input is a `DatraFrame`. This attribute should be a NumPy
array of column names. Lastly, implement the `get_features_names_out()` method: it should have one optional `input_features=None` argument. If passed the method
should check that its length matches `n_features_in_`, and it should by returned. If `input_features` is none, then the method should either return `feature_names_in_` if
it is defined or `np.array(["x0", "x1", ...])` with length `n_features_in_` otherwise.

In [34]:
class StandardScalerClone(BaseEstimator, TransformerMixin):

    def __init__(self, with_mean=True, with_std=True):
        self.with_mean = with_mean
        self.with_std = with_std

    def fit(self, X, y=None):
        X = check_array(X)
        self.n_features_names_in_ = X.shape[1]
        self._mean = X.mean(axis=1)
        self._std = X.std(axis=1)
        return self

    def transform(self, X):
        check_is_fitted(self)
        scaledX = X.copy()

        if self.with_mean:
            scaledX = X - self._mean
        if self.with_std:
            scaledX = scaledX / self._std
        return scaledX

    def inverse_transform(self, transformedX):
        check_is_fitted(self)
        X = transformedX.copy()

        if self.with_std:
            X = transformedX * self._std
        if self.with_mean:
            X = X + self._mean
        return X

    def get_features_names_out(self, input_features=None):
        if input_features and len(input_features) == n_features_in_:
            return input_features
        else:
            return features_names_in_

standard_scaler_clone = StandardScalerClone()

In [35]:
from sklearn.utils.estimator_checks import check_estimator

check_estimator(standard_scaler_clone)

ValueError: operands could not be broadcast together with shapes (20,5) (20,) 